In [1]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path

In [2]:
DEVICE = "cpu"
# DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", DEVICE)

Device: cpu


## Configuration

In [3]:
BATCH_SIZE = 32
BLOCK_SIZE = 128        # Context length
D_MODEL = 128            # Embedding size
N_HEADS = 2             # Attention heads
N_LAYERS = 2            # Transformer blocks

LEARNING_RATE = 3e-4    # 1e-3 converges in 5000 iters; 3e-4 is still falling at the end
MAX_ITERS = 5000
EVAL_INTERVAL = 500     # how often to report train/val loss
EVAL_ITERS = 100        # batches averaged per loss estimate

## Dataset

In [4]:
data_path = Path("../datasets/tinyshakespeare.txt")
text = data_path.read_text(encoding="utf-8")
print("Characters:", len(text))

Characters: 1115393


## Tokenizer

In [5]:
chars = sorted(set(text))
VOCAB_SIZE = len(chars)

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}

def encode(s):
    return [stoi[c] for c in s]


def decode(ids):
    return "".join(itos[i] for i in ids)

data = torch.tensor(encode(text), dtype=torch.long)

print(chars)
print(f"VOCAB_SIZE = {VOCAB_SIZE}")
print("Encoded shape:", data)    

['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
VOCAB_SIZE = 65
Encoded shape: tensor([18, 47, 56,  ..., 52, 45,  8])


## Train test split

In [6]:
split = int(0.8 * len(data))

train_data = data[:split]
val_data = data[split:]

print("train:", train_data.shape)
print("val  :", val_data.shape)

train: torch.Size([892314])
val  : torch.Size([223079])


## Batching

Language modelling target = the input shifted one step left: predict token `t+1` from
tokens `0..t`. Each batch samples random windows so the model sees the whole corpus.

In [7]:
torch.manual_seed(1337)


def get_batch(split):
    d = train_data if split == "train" else val_data

    # random starting offsets, one per sequence in the batch
    ix = torch.randint(len(d) - BLOCK_SIZE - 1, (BATCH_SIZE,))

    x = torch.stack([d[i:i + BLOCK_SIZE] for i in ix])              # (B, T)
    y = torch.stack([d[i + 1:i + BLOCK_SIZE + 1] for i in ix])      # (B, T) = x shifted by 1

    return x.to(DEVICE), y.to(DEVICE)


xb, yb = get_batch("train")

print("x:", xb.shape)
print("y:", yb.shape)
print("input :", repr(decode(xb[0, :40].tolist())))
print("target:", repr(decode(yb[0, :40].tolist())))

x: torch.Size([32, 128])
y: torch.Size([32, 128])
input : 'vens.\nTeach thy necessity to reason thus'
target: 'ens.\nTeach thy necessity to reason thus;'


## Model

In [8]:
class GPTTiny(nn.Module):
    def __init__(self, vocab_size, d_model, block_size):
        super().__init__()
        self.block_size = block_size
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(block_size, d_model)
        self.lm_head = nn.Linear(d_model, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding(idx)                                    # (B, T, d_model)
        pos_emb = self.position_embedding(torch.arange(T, device=idx.device))  # (T, d_model)
        x = tok_emb + pos_emb                                                  # (B, T, d_model)
        logits = self.lm_head(x)                                               # (B, T, vocab_size)

        if targets is None:
            return logits, None

        # cross_entropy wants (N, C) and (N), so flatten batch and time together
        loss = F.cross_entropy(logits.view(B * T, -1), targets.view(B * T))
        
        return logits, loss


In [9]:
model = GPTTiny(vocab_size=VOCAB_SIZE, d_model=D_MODEL, block_size=BLOCK_SIZE).to(DEVICE)

print(model)
print("params:", sum(p.numel() for p in model.parameters()))

GPTTiny(
  (token_embedding): Embedding(65, 128)
  (position_embedding): Embedding(128, 128)
  (lm_head): Linear(in_features=128, out_features=65, bias=True)
)
params: 33089


In [10]:
# sanity check: one forward pass with targets
logits, loss = model(xb, yb)

print("input :", xb.shape)
print("logits:", logits.shape)
print("loss  :", round(loss.item(), 4))
print("random-guess baseline:", round(math.log(VOCAB_SIZE), 4))

input : torch.Size([32, 128])
logits: torch.Size([32, 128, 65])
loss  : 4.4915
random-guess baseline: 4.1744


## Training


In [11]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

for it in range(MAX_ITERS + 1):
    if it % EVAL_INTERVAL == 0:
        # average loss over several batches - far less noisy than a single batch
        model.eval()
        avg = {}
        with torch.no_grad():
            for split in ("train", "val"):
                losses = torch.zeros(EVAL_ITERS)
                for k in range(EVAL_ITERS):
                    X, Y = get_batch(split)
                    _, eval_loss = model(X, Y)
                    losses[k] = eval_loss.item()
                avg[split] = losses.mean().item()
        model.train()

        print(f"iter {it:5d} | train {avg['train']:.4f} | val {avg['val']:.4f}")

    xb, yb = get_batch("train")

    _, loss = model(xb, yb)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print("done")

iter     0 | train 4.4725 | val 4.4602
iter   500 | train 2.6650 | val 2.6950
iter  1000 | train 2.5429 | val 2.5799
iter  1500 | train 2.5049 | val 2.5413
iter  2000 | train 2.4914 | val 2.5239
iter  2500 | train 2.4758 | val 2.5189
iter  3000 | train 2.4740 | val 2.5118
iter  3500 | train 2.4683 | val 2.5098
iter  4000 | train 2.4639 | val 2.5044
iter  4500 | train 2.4627 | val 2.5008
iter  5000 | train 2.4573 | val 2.5021
done
